In [1]:
import os
os.chdir("..")
os.chdir("..")

In [4]:
import timm
import torch
from torch.utils.data import DataLoader
from src.classification.data.for_data import FoRDataset
from src.classification.model.onnx.exporter import export_onnx
from src.classification.model.onnx.calibration import CNNCalibrationDataReader
from src.classification.model.onnx.quantizer import quantize_int8

In [ ]:
model = timm.create_model(
    "timm/convnextv2_base.fcmae_ft_in1k",
    pretrained=False,
    num_classes=2,
    in_chans=1,
    cache_dir=r"D:\Project\Synthetic Speech Recognizer\.cache"
)

checkpoint = torch.load(
    r"D:\Project\Synthetic Speech Recognizer\models\convnext_base\best_model.pt",
    map_location="cpu"
)

model.load_state_dict(
    checkpoint,
    strict=True
)

model = model.eval().cpu()

In [5]:
val_dataset = FoRDataset(
    root_path=r"D:\Project\Synthetic Speech Recognizer\datasets\ASVLibri",
    split="val",
    augment=False,
)

calibration_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
first_batch = next(iter(calibration_loader))
example = first_batch[0][:1]

example = example.detach().cpu().float()

Example shape: torch.Size([1, 1, 128, 188])
Example dtype: torch.float32


In [ ]:
export_onnx(
    model=model,
    example_input=example,
    save_path=r"D:\Project\Synthetic Speech Recognizer\models\convnext_base\best_model_fp32.onnx",
)

In [ ]:
calibration_reader = CNNCalibrationDataReader(
    dataloader=calibration_loader,
    input_name="input",
    max_samples=500,
)

In [ ]:
quantize_int8(
    fp32_path=r"D:\Project\Synthetic Speech Recognizer\models\convnext_base\best_model_fp32.onnx",
    int8_path=r"D:\Project\Synthetic Speech Recognizer\models\convnext_base\best_model_int8.onnx",,
    calibration_reader=calibration_reader,
)